# Task 5: Per-Tensor and Per-Channel Quantization

## Objective:
Compare per-tensor and per-output-channel symmetric quantization for convolution weights. Understand why per-channel quantization preserves accuracy better when channels have different value ranges.

In [2]:
#import the req libs

import numpy as np

In [6]:
np.random.seed(42)

conv_weights = np.random.randn(
    8, 3, 3, 3
).astype(np.float32)

# Create range imbalance
conv_weights[0] *= 0.1
conv_weights[1] *= 0.5
conv_weights[6] *= 5.0
conv_weights[7] *= 10.0



Part A: Per-Tensor Symmetric Quantization

One scale for the entire tensor:

scale = max(|weight_tensor|) / 127



In [7]:
#PART A


def per_tensor_quantize(tensor):

    # Single scale for entire tensor
    # Return: (quantized, dequantized, scale)


    max_abs = np.max(np.abs(tensor))
    scale = max_abs / 127
    quantized = np.round(tensor / scale)
    quantized = np.clip(quantized,-127,127)
    quantized = quantized.astype(np.int8)
    dequantized = quantized.astype(np.float32) * scale

    return quantized, dequantized, scale

Part B: Per-Output-Channel Symmetric Quantization

One scale per output channel (axis 0):

scale[c] = max(|weight_tensor[c]|) / 127



In [9]:


def per_channel_quantize(tensor):

    # One scale per output channel
    # Return: (quantized, dequantized, scales_array)

    quantized = np.zeros_like(tensor, dtype=np.int8)
    dequantized = np.zeros_like(tensor, dtype=np.float32)
    scales = np.zeros(tensor.shape[0], dtype=np.float32)

    for c in range(tensor.shape[0]):

        channel = tensor[c]
        max_abs = np.max(np.abs(channel))
        scale = max_abs / 127
        scales[c] = scale
        q = np.round(channel / scale)
        q = np.clip(q,-127,127)
        quantized[c] = q.astype(np.int8)
        dequantized[c] = (quantized[c].astype(np.float32)* scale)

    return quantized, dequantized, scales


In [10]:
def calculate_mae(original, reconstructed):
    return np.mean(np.abs(original - reconstructed))

In [12]:
tensor_q, tensor_deq, tensor_scale = per_tensor_quantize(conv_weights)
channel_q, channel_deq, channel_scales = per_channel_quantize(conv_weights)

In [19]:
tensor_mae_list = []
channel_mae_list = []

print("Channel Comparison")
print("--------------------------------------------------------------------------------")

for c in range(8):

    original = conv_weights[c]

    tensor_mae = calculate_mae(original, tensor_deq[c])
    channel_mae = calculate_mae(original, channel_deq[c])

    tensor_mae_list.append(tensor_mae)
    channel_mae_list.append(channel_mae)

    better = "Per-Channel" if channel_mae <= tensor_mae else "Per-Tensor"

    print(f"Channel {c}")
    print(f"Range           : ({original.min():.3f}, {original.max():.3f})")
    print(f"Per-Tensor Scale: {tensor_scale:.6f}")
    print(f"Per-Tensor MAE  : {tensor_mae:.6f}")
    print(f"Per-Ch Scale    : {channel_scales[c]:.6f}")
    print(f"Per-Ch MAE      : {channel_mae:.6f}")
    print(f"Better Method   : {better}")
    print("-" * 80)

print("Average Results")
print(f"Average Per-Tensor MAE : {np.mean(tensor_mae_list):.6f}")
print(f"Average Per-Channel MAE: {np.mean(channel_mae_list):.6f}")

Channel Comparison
--------------------------------------------------------------------------------
Channel 0
Range           : (-0.191, 0.158)
Per-Tensor Scale: 0.303365
Per-Tensor MAE  : 0.071464
Per-Ch Scale    : 0.001507
Per-Ch MAE      : 0.000326
Better Method   : Per-Channel
--------------------------------------------------------------------------------
Channel 1
Range           : (-0.980, 0.926)
Per-Tensor Scale: 0.303365
Per-Tensor MAE  : 0.072564
Per-Ch Scale    : 0.007715
Per-Ch MAE      : 0.001661
Better Method   : Per-Channel
--------------------------------------------------------------------------------
Channel 2
Range           : (-2.620, 1.565)
Per-Tensor Scale: 0.303365
Per-Tensor MAE  : 0.072175
Per-Ch Scale    : 0.020628
Per-Ch MAE      : 0.005373
Better Method   : Per-Channel
--------------------------------------------------------------------------------
Channel 3
Range           : (-1.464, 1.886)
Per-Tensor Scale: 0.303365
Per-Tensor MAE  : 0.071594
Per-Ch Scale 

| Channel | Range (Min, Max) | Per-Tensor Scale | Per-Tensor MAE | Per-Ch Scale | Per-Ch MAE | Better |
|:------:|:----------------:|-----------------:|---------------:|-------------:|-----------:|:-------:|
| 0 | (-0.191, 0.158) | 0.303365 | 0.071464 | 0.001507 | 0.000326 | Per-Channel |
| 1 | (-0.980, 0.926) | 0.303365 | 0.072564 | 0.007715 | 0.001661 | Per-Channel |
| 2 | (-2.620, 1.565) | 0.303365 | 0.072175 | 0.020628 | 0.005373 | Per-Channel |
| 3 | (-1.464, 1.886) | 0.303365 | 0.071594 | 0.014852 | 0.003731 | Per-Channel |
| 4 | (-1.919, 2.463) | 0.303365 | 0.070571 | 0.019396 | 0.003995 | Per-Channel |
| 5 | (-1.607, 1.866) | 0.303365 | 0.068939 | 0.014691 | 0.003901 | Per-Channel |
| 6 | (-5.354, 13.601) | 0.303365 | 0.077800 | 0.107093 | 0.027714 | Per-Channel |
| 7 | (-15.148, 38.527) | 0.303365 | 0.064038 | 0.303365 | 0.064038 | Per-Channel |
| **Average** | - | **0.303365** | **0.071143** | **0.061781** | **0.013842** | **Per-Channel** |

## Analysis Questions

### 1. Why does per-channel quantization usually produce lower error than per-tensor quantization?

Per-channel quantization usually produces lower error because each output channel has its own quantization scale. Since different channels have different value ranges, using separate scales preserves the values more accurately. In this experiment, the average MAE reduced from 0.071143 (Per-Tensor) to 0.013842 (Per-Channel), showing that per-channel quantization gives better accuracy.

### 2. Which channels benefit the most from per-channel quantization, and why?

Channels with smaller value ranges benefit the most because they can use a much smaller quantization scale. In this experiment, Channel 0 showed the largest improvement. Its MAE reduced from 0.071464 to 0.000326. Similarly, Channels 1 to 6 also showed much lower errors because each channel used its own suitable scale.

### 3. Why do channels with large value ranges show a smaller difference between per-tensor and per-channel quantization?

Channels with large value ranges usually decide the global scale used in per-tensor quantization. Therefore, their per-channel scale becomes almost the same as the per-tensor scale. In this experiment,Channel 7 had the largest value range, and both methods used the same scale 0.303365 and produced the same MAE 0.064038. Hence, there was almost no difference.

### 4. What is the main trade-off between per-tensor and per-channel quantization?

The main trade-off is between accuracy and simplicity. Per-tensor quantization is simpler, faster, and requires storing only one scale, but it usually has higher quantization error. Per-channel quantization gives better accuracy by using a separate scale for each channel, but it requires storing multiple scales and slightly increases computation.